## Colab / Local Dependency Setup

Jalankan cell ini sekali jika kernel belum punya dependency. Di local repo, dependency utama sudah ditambahkan ke `pyproject.toml` melalui `uv add --no-cache yt-dlp`.


In [ ]:
%pip install -q yt-dlp pandas pyarrow


# YouTube Judol Video Seed Discovery with yt-dlp

## tl;dr

Notebook ini mencari **kandidat video YouTube bertema judol** memakai `yt-dlp` sebagai discovery engine. Outputnya adalah metadata video dan ranking kandidat seed, bukan komentar dan bukan file video. Seed yang lolos review dapat dipakai di `yt_scraper.ipynb` untuk mengambil komentar melalui YouTube Data API v3.

## Context & Methods

Paper YouTube judol yang sudah kita analisis memakai pendekatan seed video/channel dan keyword evidence. Karena komentar YouTube 2026 dari video umum sulit menghasilkan promosi judol, notebook ini mencari video dengan judul/deskripsi yang eksplisit seperti `slot gacor`, `bonus new member`, `deposit`, `maxwin`, dan `rtp live`.

### Key Assumptions

- `yt-dlp` hanya dipakai untuk discovery metadata video, bukan download media.
- Hasil discovery adalah candidate seed yang tetap perlu review manual.
- Label default tetap `perlu_review`; video tidak otomatis menjadi ground-truth tanpa pengecekan.
- Metadata video bisa menjadi dataset teks tersendiri (`youtube_video_metadata`) dan juga sumber seed komentar.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import pandas as pd
import yt_dlp
from yt_dlp.utils import DownloadError, ExtractorError

pd.set_option("display.max_colwidth", 180)


## Setup

Atur query dan batas hasil di sini. Untuk pilot awal, gunakan jumlah kecil dulu agar review manual tetap masuk akal.


In [ ]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "notebooks").exists() and (CURRENT_DIR / "docs").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.name == "notebooks") and (CURRENT_DIR.parent / "docs").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

OUTPUT_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_RESULTS_PER_QUERY = 40
RUN_DISCOVERY = True

DISCOVERY_QUERIES = [
    "slot gacor hari ini",
    "slot online gacor",
    "bonus new member slot",
    "deposit dana slot",
    "link slot gacor",
    "rtp live slot hari ini",
    "maxwin slot pragmatic",
    "pragmatic play slot gacor",
    "mahjong ways gacor",
    "togel online terpercaya",
]

BASELINE_KEYWORDS = [
    "judi", "judi online", "judol", "slot", "togel", "casino", "kasino",
    "gacor", "maxwin", "jackpot", "bonus", "bonus new member",
    "deposit", "depo", "wd", "withdraw", "daftar", "link",
    "link alternatif", "situs", "login", "member", "pragmatic", "scatter",
    "mahjong", "rtp", "freebet",
]

STRONG_TITLE_TERMS = [
    "slot gacor", "slot online", "bonus new member", "deposit dana",
    "maxwin", "rtp live", "pragmatic play", "mahjong ways", "togel online",
]

CONTACT_PATTERNS = [
    r"https?://",
    r"www\.",
    r"t\.me/",
    r"telegram",
    r"whatsapp",
    r"\bwa\b",
]

print({
    "project_root": str(PROJECT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "queries": len(DISCOVERY_QUERIES),
    "max_results_per_query": MAX_RESULTS_PER_QUERY,
    "run_discovery": RUN_DISCOVERY,
})


## Helper Functions

Helper di bawah ini menjaga discovery tetap auditable: metadata diambil tanpa download, query provenance disimpan, dan scoring hanya dipakai untuk prioritas review.


In [ ]:
ASCII_WORD_RE = re.compile(r"[a-z0-9]+")
CONTACT_RE = re.compile("|".join(CONTACT_PATTERNS), re.IGNORECASE)
ZERO_WIDTH_CHARS = {"\u200b", "\u200c", "\u200d", "\ufeff"}


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def normalize_text(text: str) -> str:
    lowered = (text or "").casefold()
    normalized = unicodedata.normalize("NFKC", lowered)
    return re.sub(r"\s+", " ", normalized).strip()


def compact_alnum_text(text: str) -> str:
    return "".join(ASCII_WORD_RE.findall(normalize_text(text)))


def find_keyword_matches(text: str, keywords: Iterable[str]) -> list[str]:
    normalized = normalize_text(text)
    compact = compact_alnum_text(text)
    matches = []
    for keyword in keywords:
        key_norm = normalize_text(keyword)
        key_compact = re.sub(r"[^a-z0-9]+", "", key_norm)
        if key_norm and key_norm in normalized:
            matches.append(keyword)
        elif key_compact and key_compact in compact:
            matches.append(keyword)
    return sorted(set(matches))


def detect_unicode_flags(text: str) -> list[str]:
    flags = []
    if any(ord(ch) > 127 for ch in text or ""):
        flags.append("non_ascii")
    if any(ch in ZERO_WIDTH_CHARS for ch in text or ""):
        flags.append("zero_width")
    if re.search(r"[a-zA-Z][0-9][a-zA-Z]|[0-9][a-zA-Z]|[a-zA-Z][0-9]", text or ""):
        flags.append("number_substitution")
    if re.search(r"(?:[a-zA-Z0-9]\s+){2,}[a-zA-Z0-9]", text or ""):
        flags.append("irregular_spacing")
    return sorted(set(flags))


def stable_row_id(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()[:16]


def extract_video_entries(search_payload: dict) -> list[dict]:
    entries = search_payload.get("entries") or []
    return [entry for entry in entries if isinstance(entry, dict) and entry.get("id")]


def search_youtube_videos(query: str, max_results: int) -> list[dict]:
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        "extract_flat": "in_playlist",
        "ignoreerrors": True,
        "noplaylist": True,
    }
    search_url = f"ytsearch{max_results}:{query}"
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        payload = ydl.extract_info(search_url, download=False)
        return extract_video_entries(payload or {})


def build_video_row(entry: dict, query: str) -> dict:
    video_id = entry.get("id", "")
    title = entry.get("title", "") or ""
    description = entry.get("description", "") or ""
    combined_text = f"{title} {description}"
    keyword_matches = find_keyword_matches(combined_text, BASELINE_KEYWORDS)
    strong_matches = find_keyword_matches(title, STRONG_TITLE_TERMS)
    unicode_flags = detect_unicode_flags(combined_text)
    has_contact = bool(CONTACT_RE.search(combined_text))
    candidate_score = (len(keyword_matches) * 1) + (len(strong_matches) * 2) + int(has_contact) + int(bool(unicode_flags))

    return {
        "row_id": stable_row_id(f"youtube:{video_id}:{query}"),
        "platform": "youtube",
        "source_type": "yt_dlp_search",
        "source_seed": query,
        "collection_query": f"ytsearch{MAX_RESULTS_PER_QUERY}:{query}",
        "video_id": video_id,
        "video_url": entry.get("url") or f"https://www.youtube.com/watch?v={video_id}",
        "title": title,
        "description": description,
        "channel": entry.get("channel") or entry.get("uploader") or "",
        "channel_id": entry.get("channel_id") or "",
        "uploader_id": entry.get("uploader_id") or "",
        "duration": entry.get("duration"),
        "view_count": entry.get("view_count"),
        "live_status": entry.get("live_status") or "",
        "keyword_matches": keyword_matches,
        "strong_title_matches": strong_matches,
        "candidate_flag_keyword": bool(keyword_matches),
        "candidate_flag_contact": has_contact,
        "candidate_flag_unicode": bool(unicode_flags),
        "unicode_obfuscation_flags": unicode_flags,
        "candidate_score": candidate_score,
        "label": "perlu_review",
        "label_source": "unlabeled",
        "paper_basis": "keyword-based discovery + YouTube seed-video strategy from judol dataset papers",
        "collected_at": utc_now_iso(),
        "notes": "",
    }


## Offline Smoke Check

Cell ini memastikan scoring lokal bekerja tanpa membuka koneksi ke YouTube.


In [ ]:
sample_entry = {
    "id": "demo123",
    "url": "https://www.youtube.com/watch?v=demo123",
    "title": "Bonus new member slot gacor deposit dana hari ini",
    "description": "Link alternatif via Telegram tersedia.",
    "channel": "Demo Channel",
}
smoke_row = build_video_row(sample_entry, "slot gacor")
assert smoke_row["candidate_flag_keyword"]
assert smoke_row["candidate_flag_contact"]
assert "slot" in smoke_row["keyword_matches"]
assert smoke_row["candidate_score"] > 0
smoke_row


## Run Discovery

Jalankan discovery untuk semua query. Jika YouTube membatasi request, kurangi `MAX_RESULTS_PER_QUERY` atau jumlah query.


In [ ]:
discovery_rows = []
query_errors = []

if RUN_DISCOVERY:
    for query in DISCOVERY_QUERIES:
        try:
            entries = search_youtube_videos(query, MAX_RESULTS_PER_QUERY)
            discovery_rows.extend(build_video_row(entry, query) for entry in entries)
            print({"query": query, "entries": len(entries)})
        except (DownloadError, ExtractorError, RuntimeError) as error:
            query_errors.append({
                "query": query,
                "error": str(error),
                "collected_at": utc_now_iso(),
            })
else:
    print("Discovery skipped because RUN_DISCOVERY=False.")

df_discovery_raw = pd.DataFrame(discovery_rows)
df_query_errors = pd.DataFrame(query_errors)

if not df_discovery_raw.empty:
    df_discovery = (
        df_discovery_raw
        .sort_values(["candidate_score", "view_count"], ascending=[False, False], na_position="last")
        .drop_duplicates(subset=["video_id"], keep="first")
        .reset_index(drop=True)
    )
else:
    df_discovery = df_discovery_raw.copy()

print({
    "raw_rows": len(df_discovery_raw),
    "deduped_rows": len(df_discovery),
    "query_errors": len(df_query_errors),
})

if not df_query_errors.empty:
    display(df_query_errors)


## Review Candidates

Gunakan tabel ini untuk memilih seed video. Kolom `label` sengaja tetap `perlu_review`.


In [ ]:
review_columns = [
    "video_id", "video_url", "title", "channel", "view_count", "live_status",
    "source_seed", "keyword_matches", "strong_title_matches",
    "candidate_flag_contact", "candidate_flag_unicode", "candidate_score", "label",
]

if df_discovery.empty:
    print("No discovery rows yet.")
else:
    display(df_discovery[review_columns].head(30))


## Export

Output utama adalah CSV kandidat seed. `selected_seed_template` membantu memindahkan video terpilih ke `SEED_VIDEOS` pada notebook komentar API.


In [ ]:
DISCOVERY_OUTPUT = OUTPUT_DIR / "youtube_ytdlp_video_seed_candidates.csv"
SEED_TEMPLATE_OUTPUT = OUTPUT_DIR / "youtube_ytdlp_seed_template.csv"
ERROR_OUTPUT = OUTPUT_DIR / "youtube_ytdlp_query_errors.csv"

if not df_discovery.empty:
    df_discovery.to_csv(DISCOVERY_OUTPUT, index=False)
    seed_template = df_discovery.head(50).copy()
    seed_template = seed_template.assign(
        seed_type="video_id",
        seed_value=seed_template["video_id"],
        seed_reason="yt-dlp keyword discovery candidate; requires manual review",
        collection_query="manual_review_from_ytdlp:" + seed_template["source_seed"].astype(str),
    )[[
        "video_id", "seed_type", "seed_value", "seed_reason",
        "paper_basis", "collection_query", "title", "video_url", "candidate_score",
    ]]
    seed_template.to_csv(SEED_TEMPLATE_OUTPUT, index=False)
else:
    seed_template = pd.DataFrame()

if not df_query_errors.empty:
    df_query_errors.to_csv(ERROR_OUTPUT, index=False)

print({
    "discovery_output": str(DISCOVERY_OUTPUT),
    "seed_template_output": str(SEED_TEMPLATE_OUTPUT),
    "error_output": str(ERROR_OUTPUT) if not df_query_errors.empty else None,
    "deduped_rows": len(df_discovery),
})


## Results

Ringkasan ini membantu melihat apakah query terlalu luas atau terlalu sempit.


In [ ]:
if df_discovery.empty:
    print("No results to summarize.")
else:
    print({
        "videos": int(len(df_discovery)),
        "keyword_candidates": int(df_discovery["candidate_flag_keyword"].sum()),
        "contact_candidates": int(df_discovery["candidate_flag_contact"].sum()),
        "unicode_candidates": int(df_discovery["candidate_flag_unicode"].sum()),
        "top_query_counts": df_discovery["source_seed"].value_counts().head(10).to_dict(),
    })
    display(df_discovery.groupby("source_seed").agg(
        videos=("video_id", "count"),
        avg_score=("candidate_score", "mean"),
        max_score=("candidate_score", "max"),
    ).sort_values(["max_score", "videos"], ascending=False))


## Takeaways

- `yt-dlp` discovery membantu menemukan video dengan judul/deskripsi judol yang mungkin sulit ditemukan dari komentar umum.
- Hasil ini sebaiknya direview manual sebelum dijadikan seed komentar.
- Untuk dataset final, pisahkan `youtube_video_metadata` dari `youtube_comments` agar label dan sumber data tetap jelas.
